In [1]:
import importlib.util
import random, os, subprocess, sys

def ensure_package(import_name, package_name=None):
    if importlib.util.find_spec(import_name) is None:
        package_name = package_name or import_name
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        importlib.invalidate_caches()

ensure_package("nltk")

import numpy as np
import pandas as pd
import nltk
from sklearn.metrics import f1_score

In [2]:
seed = 11037
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)

In [3]:
device = "cpu"

In [4]:
import os
import pandas as pd
from pathlib import Path

REQUIRED_DATA_FILES = ("train.csv", "test.csv")

def has_data_files(base):
    return all((base / name).exists() for name in REQUIRED_DATA_FILES)

def find_data_dir():
    env_dir = os.environ.get("POLARITY_DATA_DIR")
    candidates = [
        Path(env_dir) if env_dir else None,
        Path("/kaggle/input/polarity-aicc-round-7"),
        Path("/kaggle/input/competitions/polarity-aicc-round-7"),
        Path.cwd(),
        Path.cwd() / "baseline-polarity-aicc",
        Path.cwd().parent / "baseline-polarity-aicc",
        Path.cwd() / "data",
        Path.cwd().parent / "data",
    ]

    searched = []
    for base in candidates:
        if base is None:
            continue
        base = base.expanduser()
        searched.append(str(base))
        if has_data_files(base):
            return base

    raise FileNotFoundError(
        "Could not find train.csv and test.csv. Set POLARITY_DATA_DIR or place both files next to this notebook. "
        f"Searched: {searched}"
    )

DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else DATA_DIR

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print(f"Loaded data from: {DATA_DIR.resolve()}")

print(f"train {train.shape}, test {test.shape}, label counts {train['label'].value_counts().to_dict()}")
train.head()

train (50, 3), test (686, 3), label counts {0: 25, 1: 25}


,w1,w2,label
0,cash,money,0
1,grab,take,0
2,irrational,rational,1
3,defend,guard,0
4,instructor,teacher,0


In [ ]:
# =====================================================================
# Fast synonym (0) vs antonym (1) features.
#
# This replaces the slow BERT prompt/fine-tuning path with lexical
# evidence that is much faster on CPU:
#   * WordNet synonym / antonym expansion, including similar-to links.
#   * Morphology for negating prefixes such as un-, in-, dis-, non-.
#   * A small list of common complementary pairs not encoded as WordNet
#     lemma antonyms, e.g. buyer/seller and listener/speaker.
# =====================================================================
import re
from functools import lru_cache

import nltk
import numpy as np
from nltk.corpus import wordnet as wn

for package, resource in (("wordnet", "corpora/wordnet"), ("omw-1.4", "corpora/omw-1.4")):
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True)


def clean(word):
    return re.sub(r"\s+", "_", str(word).strip().lower())


def human(word):
    return clean(word).replace("_", " ")


def compact(word):
    return re.sub(r"[^a-z]", "", human(word))


@lru_cache(None)
def variants(word):
    word = clean(word)
    out = {word}
    for pos in (None, wn.NOUN, wn.VERB, wn.ADJ, wn.ADV):
        morphed = wn.morphy(word, pos) if pos else wn.morphy(word)
        if morphed:
            out.add(morphed)
    return tuple(sorted(out))


@lru_cache(None)
def synsets_for(word):
    out = []
    seen = set()
    for variant in variants(word):
        for synset in wn.synsets(variant):
            if synset.name() not in seen:
                out.append(synset)
                seen.add(synset.name())
    return tuple(out)


def names_from_synset(synset):
    return {lemma.name().lower().replace("_", " ") for lemma in synset.lemmas()}


@lru_cache(None)
def related_names(word):
    names = set()
    related_synsets = []
    for synset in synsets_for(word):
        related_synsets.append(synset)
        related_synsets.extend(synset.similar_tos())
        related_synsets.extend(synset.also_sees())
        related_synsets.extend(synset.attributes())

    seen = set()
    for synset in related_synsets:
        if synset.name() in seen:
            continue
        seen.add(synset.name())
        names.update(names_from_synset(synset))
    return frozenset(names)


@lru_cache(None)
def antonym_names(word):
    names = set()
    base_synsets = []
    for synset in synsets_for(word):
        base_synsets.append(synset)
        base_synsets.extend(synset.similar_tos())
        base_synsets.extend(synset.also_sees())
        base_synsets.extend(synset.attributes())

    for name in list(related_names(word))[:100]:
        base_synsets.extend(synsets_for(name))

    seen = set()
    for synset in base_synsets:
        if synset.name() in seen:
            continue
        seen.add(synset.name())
        for lemma in synset.lemmas():
            for antonym in lemma.antonyms():
                names.add(antonym.name().lower().replace("_", " "))
                antonym_synset = antonym.synset()
                names.update(names_from_synset(antonym_synset))
                for linked in antonym_synset.similar_tos() + antonym_synset.also_sees() + antonym_synset.attributes():
                    names.update(names_from_synset(linked))
    return frozenset(names)


NEG_PREFIXES = ("un", "in", "im", "ir", "il", "dis", "non", "mis", "anti", "a", "de", "counter")
OPP_PREFIX_PAIRS = (
    ("in", "ex"), ("im", "ex"), ("en", "dis"), ("in", "de"),
    ("inter", "exter"), ("pro", "anti"), ("over", "under"),
    ("up", "down"), ("pre", "post"), ("fore", "back"),
)


def strip_prefix(word, prefix):
    return word[len(prefix):] if word.startswith(prefix) and len(word) > len(prefix) + 2 else None


def morphology_antonym(a, b):
    a, b = compact(a), compact(b)
    if a == b:
        return 0

    short, long = sorted((a, b), key=len)
    for prefix in NEG_PREFIXES:
        if long == prefix + short:
            return 1

    for suffix_a, suffix_b in (("less", "ful"), ("ful", "less")):
        if a.endswith(suffix_a) and b.endswith(suffix_b) and len(a) > 4 and a[:-len(suffix_a)] == b[:-len(suffix_b)]:
            return 1

    for prefix_a, prefix_b in OPP_PREFIX_PAIRS:
        for left, right in ((a, b), (b, a)):
            stem_left = strip_prefix(left, prefix_a)
            stem_right = strip_prefix(right, prefix_b)
            if stem_left and stem_right and stem_left == stem_right and len(stem_left) >= 3:
                return 1
    return 0


CURATED_ANTONYM_PAIRS = {
    ("punish", "reward"), ("guest", "host"), ("hero", "villain"),
    ("create", "destroy"), ("buyer", "seller"), ("buy", "sell"),
    ("borrow", "lend"), ("give", "take"), ("teacher", "student"),
    ("leader", "follower"), ("parent", "child"), ("predator", "prey"),
    ("winner", "loser"), ("win", "lose"), ("victory", "defeat"),
    ("success", "failure"), ("friend", "enemy"), ("ally", "enemy"),
    ("master", "servant"), ("employer", "employee"), ("import", "export"),
    ("arrival", "departure"), ("arrive", "depart"), ("entrance", "exit"),
    ("enter", "exit"), ("question", "answer"), ("cause", "effect"),
    ("problem", "solution"), ("attack", "defend"), ("offense", "defense"),
    ("male", "female"), ("man", "woman"), ("boy", "girl"),
    ("king", "queen"), ("husband", "wife"), ("brother", "sister"),
    ("up", "down"), ("left", "right"), ("top", "bottom"),
    ("north", "south"), ("east", "west"), ("first", "last"),
    ("beginning", "end"), ("start", "finish"), ("open", "close"),
    ("empty", "full"), ("presence", "absence"), ("include", "exclude"),
    ("inclusion", "exclusion"), ("encourage", "discourage"),
    ("accept", "reject"), ("acceptance", "rejection"),
    ("clarity", "confusion"), ("listener", "speaker"), ("vice", "virtue"),
}
CURATED_ANTONYM_PAIRS |= {(b, a) for a, b in list(CURATED_ANTONYM_PAIRS)}


def curated_antonym(a, b):
    return int((human(a), human(b)) in CURATED_ANTONYM_PAIRS or (compact(a), compact(b)) in CURATED_ANTONYM_PAIRS)


def pair_features(a, b):
    a_human, b_human = human(a), human(b)
    direct_synonym = int(b_human in related_names(a) or a_human in related_names(b))
    direct_antonym = int(b_human in antonym_names(a) or a_human in antonym_names(b))
    morph_antonym = morphology_antonym(a, b)
    curated = curated_antonym(a, b)

    synsets_a = synsets_for(a)
    synsets_b = synsets_for(b)
    same_synset = 0
    max_path = 0.0
    max_wup = 0.0
    for synset_a in synsets_a:
        for synset_b in synsets_b:
            same_synset = max(same_synset, int(synset_a == synset_b))
            path_similarity = synset_a.path_similarity(synset_b)
            if path_similarity is not None:
                max_path = max(max_path, path_similarity)
            wup_similarity = synset_a.wup_similarity(synset_b)
            if wup_similarity is not None:
                max_wup = max(max_wup, wup_similarity)

    a_compact, b_compact = compact(a), compact(b)
    lexical = [
        len(a_compact),
        len(b_compact),
        abs(len(a_compact) - len(b_compact)),
        int(a_compact[:2] == b_compact[:2]),
        int(a_compact[-2:] == b_compact[-2:]),
        int(a_compact in b_compact or b_compact in a_compact),
    ]
    return [direct_synonym, direct_antonym, morph_antonym, curated, same_synset, max_path, max_wup, len(synsets_a), len(synsets_b)] + lexical


def build_features(df):
    return np.array([pair_features(row.w1, row.w2) for row in df.itertuples()], dtype=np.float32)


print("Building fast lexical features ...")
X_train = build_features(train)
X_test = build_features(test)
y = train["label"].values.astype(int)
print("Feature matrix:", X_train.shape, "train /", X_test.shape, "test")


In [ ]:
# =====================================================================
# Rule-first classifier and submission.
#
# Rules make high-confidence calls. A tiny ExtraTrees model is kept only
# as a fallback for pairs that have no synonym/antonym signal.
# =====================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score

SEED = 11037


def rule_masks(X):
    direct_synonym = X[:, 0] == 1
    direct_antonym = X[:, 1] == 1
    morph_antonym = X[:, 2] == 1
    curated_antonym_feature = X[:, 3] == 1
    same_synset = X[:, 4] == 1
    path_close = X[:, 5] >= 0.20

    antonym_rule = direct_antonym | morph_antonym | curated_antonym_feature
    synonym_rule = (direct_synonym | same_synset | path_close) & ~antonym_rule
    return antonym_rule, synonym_rule


def predict_with_rules(X, fallback_model=None):
    antonym_rule, synonym_rule = rule_masks(X)
    unknown = ~(antonym_rule | synonym_rule)
    pred = np.zeros(len(X), dtype=int)
    pred[antonym_rule] = 1
    pred[synonym_rule] = 0

    if unknown.any() and fallback_model is not None:
        pred[unknown] = fallback_model.predict(X[unknown])
    return pred, antonym_rule, synonym_rule, unknown


fallback = ExtraTreesClassifier(
    n_estimators=400,
    random_state=SEED,
    class_weight="balanced",
    max_features=None,
    min_samples_leaf=1,
)
fallback.fit(X_train, y)

train_pred, train_ant_rule, train_syn_rule, train_unknown = predict_with_rules(X_train, fallback)
train_f1 = f1_score(y, train_pred, average="macro")
print(f"Training macro-F1 check: {train_f1:.4f}")
print("Train rule coverage:", int(train_ant_rule.sum()), "antonym,", int(train_syn_rule.sum()), "synonym,", int(train_unknown.sum()), "fallback")

test_pred, test_ant_rule, test_syn_rule, test_unknown = predict_with_rules(X_test, fallback)
print("Test rule coverage:", int(test_ant_rule.sum()), "antonym,", int(test_syn_rule.sum()), "synonym,", int(test_unknown.sum()), "fallback")
print("Test predicted antonym fraction:", round(float(test_pred.mean()), 3))

submission = pd.DataFrame({"row_id": test["row_id"], "label": test_pred})
output_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(output_path, index=False)
print(f"Saved submission to {output_path.resolve()}")
print(submission["label"].value_counts(normalize=True))
